# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and name
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"  @id: {record_set['@id']}, name: {record_set.get('name', '(no name)')}")
    # List fields
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("    Fields in this record set:")
        for field in fields:
            print(f"      @id: {field['@id']}, name: {field.get('name', '(no name)')}, dataType: {field.get('dataType', '(unknown)')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Record sets and field `@id`s are referenced from the overview.

In [ ]:
# List all record set @id values for extraction
record_set_ids = [r['@id'] for r in dataset.record_sets]
dataframes = {}

# Extract data for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only make DataFrame if data exists
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} records into DataFrame for record set: {record_set_id}")

# For demo, use first record set if available
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nColumns in '{main_record_set_id}':\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No records found in the listed record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# EDA Example: Work with a numeric field in the main record set
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Find a numeric field to demonstrate:
    numeric_col = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break
    if numeric_col is None:
        # Try to coerce one of the columns to numeric
        for col in df.columns:
            try:
                coerced = pd.to_numeric(df[col], errors='coerce')
                if coerced.notna().sum() > 0:
                    df[col+'_num'] = pd.to_numeric(df[col], errors='coerce')
                    numeric_col = col+'_num'
                    break
            except:
                continue
    if numeric_col is not None:
        threshold = df[numeric_col].mean() if df[numeric_col].dtype.kind in 'biufc' else 10
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records in '{main_record_set_id}' with {numeric_col} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[numeric_col + '_normalized'] = (
            filtered_df[numeric_col] - filtered_df[numeric_col].mean()
        ) / filtered_df[numeric_col].std()
        print(f"\nNormalized '{numeric_col}' for filtered records:")
        display(filtered_df[[numeric_col, numeric_col + '_normalized']].head())

        # Group by a categorical field (if available)
        group_field = None
        for col in df.columns:
            if col != numeric_col and df[col].dtype == 'object' and df[col].nunique() < min(25, len(df)//2):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_col].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric-like field found for EDA in main record set.")
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize numeric data distributions (if any)
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_col is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_col].dropna(), bins=30, kde=True, color='steelblue')
    plt.title(f"Distribution of '{numeric_col}' in record set '{main_record_set_id}'")
    plt.xlabel(numeric_col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # If grouped, show group means
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_col, data=grouped_df)
        plt.title(f"Mean {numeric_col} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_col}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library. We loaded metadata, listed available record sets and fields by their `@id`, extracted data, performed basic EDA and visualized numeric distributions. For deeper analyses, further domain knowledge on the mass spectrometry field definitions is recommended.